In [ ]:
# Phase 8 — Training scaffold (GRPO-like REINFORCE on JSON actions)

import sys
from pathlib import Path

repo_root = Path.cwd()
if (repo_root / "env").exists() is False:
    # if launched from training/
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

print("repo_root:", repo_root)


In [ ]:
import numpy as np

from env.digital_twin_env import DigitalTwinDiabetesEnv


def make_env(seed: int):
    return DigitalTwinDiabetesEnv(seed=seed, max_steps=24)


env = make_env(0)
obs, info = env.reset()
print("patient_id", info["patient_id"], "obs", obs)


In [ ]:
# Action templates (JSON dicts). This keeps training fast & hackathon-runnable.
# Swap out `TemplatePolicy` with an LLM policy later.

action_templates = [
    {"type": "noop"},
    {"type": "start", "drug": "metformin", "dose": 1.0, "lifestyle": 0.7},
    {"type": "add", "drug": "glp1", "dose": 1.0},
    {"type": "add", "drug": "sglt2", "dose": 1.0},
    {"type": "start", "drug": "insulin", "dose": 0.7},
    {"type": "add", "drug": "dpp4", "dose": 1.0},
    {"type": "add", "drug": "sulfonylurea", "dose": 0.7},
]


class TemplatePolicy:
    def __init__(self, n_actions: int, seed: int = 0):
        self.rng = np.random.default_rng(seed)
        self.logits = np.zeros((n_actions,), dtype=np.float32)

    def probs(self) -> np.ndarray:
        x = self.logits - self.logits.max()
        p = np.exp(x)
        return p / p.sum()

    def act(self) -> int:
        p = self.probs()
        return int(self.rng.choice(len(p), p=p))

    def update_reinforce(self, chosen_idx: int, reward: float, lr: float = 0.05):
        # GRPO-like single-step: ascent on log pi(a) * reward (baseline-free)
        p = self.probs()
        grad = -p
        grad[chosen_idx] += 1.0
        self.logits += lr * reward * grad


policy = TemplatePolicy(n_actions=len(action_templates), seed=0)
print("init_probs", np.round(policy.probs(), 3))


In [ ]:
# Training loop (episodic REINFORCE over action templates)

from collections import defaultdict


def run_episode(seed: int, policy: TemplatePolicy, max_steps: int = 24):
    env = make_env(seed)
    obs, _ = env.reset(seed=seed)

    chosen = []
    rewards = []
    infos = []

    for t in range(max_steps):
        a_idx = policy.act()
        action = action_templates[a_idx]
        obs, r, term, trunc, info = env.step(action)
        chosen.append(a_idx)
        rewards.append(float(r))
        infos.append(info)
        if term or trunc:
            break

    return {
        "return": float(np.sum(rewards)),
        "steps": len(rewards),
        "chosen": chosen,
        "rewards": rewards,
        "infos": infos,
    }


history = []
for ep in range(60):
    out = run_episode(seed=ep, policy=policy)
    G = out["return"]

    # Update using episode return (credit assignment simplification)
    for a_idx in out["chosen"]:
        policy.update_reinforce(a_idx, reward=G, lr=0.01)

    history.append(out)
    if (ep + 1) % 10 == 0:
        avg = float(np.mean([h["return"] for h in history[-10:]]))
        print(f"ep={ep+1:03d} avg_return(last10)={avg:+.2f} probs={np.round(policy.probs(), 3)}")


In [ ]:
# Reward logging + plot

import matplotlib.pyplot as plt

returns = np.array([h["return"] for h in history], dtype=float)
window = 10
ma = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 3))
plt.plot(returns, alpha=0.4, label="episode return")
plt.plot(np.arange(window - 1, window - 1 + len(ma)), ma, label="moving avg")
plt.title("Training: episode returns")
plt.xlabel("episode")
plt.ylabel("return")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Optional: LLM policy stub (model loading). Not used in fast training above.
# If you want, install: pip install torch transformers

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = "distilgpt2"
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForCausalLM.from_pretrained(model_name)
    mdl.eval()

    def llm_generate_action(prompt: str, max_new_tokens: int = 64) -> str:
        x = tok(prompt, return_tensors="pt")
        with torch.no_grad():
            y = mdl.generate(**x, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.8)
        return tok.decode(y[0], skip_special_tokens=True)

    print("LLM loaded:", model_name)
except Exception as e:
    print("LLM optional block skipped:", e)
